# Assignment 08: Text-to-Text Generation
**Student Name:** Faiza Altaf  
**Submission Date:** September 02, 2026  
**Course:** Data Science & Analytics / Natural Language Processing  

---

## 1. Project Overview & Objective
This notebook demonstrates an end-to-end **Text-to-Text Generation Pipeline** for tasks such as automated text summarization, paraphrasing, and key takeaway extraction using Transformer models.

### Workflow Steps:
1. **Environment Setup:** Installing and importing Hugging Face `transformers` and `torch`.
2. **Pipeline Initialization:** Loading a pretrained sequence-to-sequence model (T5 / BART).
3. **Text Summarization & Generation:** Generating concise summaries and altered text outputs.
4. **Parameter Tuning:** Adjusting `max_length`, `num_beams`, and `temperature` for optimal output generation.
5. **Evaluation:** Displaying input text alongside generated outputs.

In [ ]:
# Step 1: Import Core Libraries
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
sns.set_palette('Set2')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

print('Libraries successfully imported!')

## 2. Pretrained Text-to-Text Model Pipeline Initialization

In [ ]:
# Step 2: Initialize T5 / BART Text-to-Text Generation Pipeline
model_name = "t5-small"
print(f"Loading pretrained model: {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)

print("Text-to-Text Generation Pipeline ready!")

## 3. Input Data & Text Generation Execution

In [ ]:
# Step 3: Define Input Articles / Prompts
input_text_1 = """
Artificial Intelligence and Machine Learning have rapidly transformed modern web development and UI/UX design. 
With the rise of Agentic AI, developers can now build interactive systems using LangChain and Retrieval-Augmented Generation (RAG) 
pipelines. These smart agents enable dynamic user interfaces that adapt to user preferences in real time, 
drastically reducing manual effort and improving application efficiency across modern frontend frameworks like React.js.
"""

input_text_2 = """
Natural Language Processing (NLP) is a subfield of computer science and artificial intelligence concerned with giving 
computers the ability to understand text and spoken words in much the same way human beings can. 
Combining computational linguistics with statistical, machine learning, and deep learning models, 
these technologies enable computers to process human language in the form of text or voice data and to comprehend 
its full meaning, complete with the speaker or writer's intent and sentiment.
"""

# Perform Text Generation (Summarization Task)
summary_1 = summarizer(input_text_1, max_length=50, min_length=20, do_sample=False)[0]['summary_text']
summary_2 = summarizer(input_text_2, max_length=50, min_length=20, do_sample=False)[0]['summary_text']

results_data = [
    {"Task": "AI in Web Dev", "Original Word Count": len(input_text_1.split()), "Generated Summary": summary_1},
    {"Task": "NLP Overview", "Original Word Count": len(input_text_2.split()), "Generated Summary": summary_2}
]

df_results = pd.DataFrame(results_data)
df_results

## 4. Hyperparameter Tuning (Beam Search & Generation Settings)

In [ ]:
# Step 4: Fine-Tuning Generation Decoding Parameters
prefix_prompt = "summarize: " + input_text_1
inputs = tokenizer(prefix_prompt, return_tensors="pt", max_length=512, truncation=True)

# Beam Search Decoding
outputs = model.generate(
    inputs["input_ids"],
    max_length=60,
    num_beams=4,
    early_stopping=True
)

tuned_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=== Tuned Generation Output (Beam Search) ===")
print(tuned_summary)

## 5. Output Length Comparison Visualization

In [ ]:
# Step 5: Visualizing Compression Ratio
df_results['Summary Word Count'] = df_results['Generated Summary'].apply(lambda x: len(x.split()))

plt.figure(figsize=(8, 5))
x = np.arange(len(df_results))
width = 0.35

plt.bar(x - width/2, df_results['Original Word Count'], width, label='Original Word Count', color='#4C72B0')
plt.bar(x + width/2, df_results['Summary Word Count'], width, label='Summary Word Count', color='#55A868')

plt.xlabel('Task')
plt.ylabel('Word Count')
plt.title('Text Compression Ratio: Original vs Generated Text')
plt.xticks(x, df_results['Task'])
plt.legend()
plt.tight_layout()
plt.show()